# 🧩 实验12：在 SO-101 数据集上训练 ACT（行为克隆）

前面的实验里，我们用策略梯度（REINFORCE）在 CartPole 上**从零**学一个策略。
这次我们换到**真实机器人**：用 **ACT**（Action Chunking Transformer）在 `lerobot/svla_so101_pickplace` 数据集上做**行为克隆（Behavior Cloning, BC）**——
即从人类遥操作的示范中，**监督式**地学「看到画面 → 输出动作」。

任务：把一块粉色乐高放进透明盒子。数据：50 条轨迹、2 路相机、6 自由度机械臂。

本实验你将：
1. **认识数据集**：看视频、看动作；
2. 理解 ACT 的网络结构；
3. 准备数据集与策略（**有填空**）；
4. 用 ACT 自带的损失训练它（**有填空**）；
5. 用训练好的策略做推理。

> 标注 `# TODO（填空 N）` 的地方需要你补全代码后才能运行。

## 0. 环境安装（第一次运行前）

需要一个 Python 3.12 的 conda 环境，装好 **PyTorch（建议 CUDA 版）**、**LeRobot** 和 **matplotlib**。
在终端（Windows 用 Anaconda PowerShell）里执行：

```powershell
# 1) 创建并激活环境
conda create -n lerobot python=3.12 -y
conda activate lerobot

# 2) 安装支持 CUDA 的 PyTorch（有 NVIDIA 显卡时）
#    先装它，避免第 3 步把 torch 降级成 CPU 版
pip install torch==2.10.0 torchvision==0.25.0 --index-url https://download.pytorch.org/whl/cu128

# 3) 安装 LeRobot（含数据集功能）与画图库
pip install "lerobot[dataset]" matplotlib
```

**说明：**
- **没有 NVIDIA 显卡？** 跳过第 2 步，直接做第 3 步即可（会装 CPU 版 PyTorch，能跑但较慢）。
- 本实验读视频用 **PyAV 后端**（代码里 `video_backend="pyav"`），PyAV 自带 FFmpeg，**无需**单独装 ffmpeg / torchcodec。
- 第一次运行会自动从 HuggingFace 下载数据集（约 86 MB）和 ResNet18 预训练权重（约 45 MB）。
- 在 Jupyter / VS Code 里，把 **kernel 选成上面这个 `lerobot` 环境**。
- 下面第一格会打印版本做自检：看到 `CUDA 可用: True` 就说明显卡可用。

In [ ]:
import torch
import lerobot
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Image as IPyImage

from lerobot.configs.types import FeatureType
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.utils.feature_utils import dataset_to_policy_features
from lerobot.utils.constants import ACTION
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.factory import make_pre_post_processors

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("lerobot 版本:", lerobot.__version__)
print("torch 版本  :", torch.__version__, "| CUDA 可用:", torch.cuda.is_available())
print("使用设备    :", device)

---
## 第一部分：认识数据集

动手训练之前，先**看看数据长什么样**：机器人在做什么、相机看到什么、动作数据是什么样子。
数据集 `svla_so101_pickplace` 记录了人用主臂遥操作 SO-101 完成「把乐高放进盒子」的 **50 次示范**。

In [ ]:
dataset_id = "lerobot/svla_so101_pickplace"
meta = LeRobotDatasetMetadata(dataset_id)
print("轨迹数:", meta.total_episodes, "| 总帧数:", meta.total_frames,
      "| fps:", meta.fps, "| 时长:", round(meta.total_frames / meta.fps / 60, 1), "分钟")
print("相机:", meta.camera_keys)
print("关节名:", meta.names["action"])

# 加载数据集（PyAV 后端；不设 delta_timestamps -> 每个样本是单帧）
explore_ds = LeRobotDataset(dataset_id, video_backend="pyav")
sample = explore_ds[0]
print("\n一个样本包含：")
for k, v in sample.items():
    if hasattr(v, "shape"):
        print(f"  {k:30s}{tuple(v.shape)}")
print("任务:", sample["task"])

### 看两路相机画面
同一时刻、两个视角：`up` 从上往下看工作台，`side` 从侧面看机械臂。ACT 训练时会**同时**用这两路画面。

In [ ]:
def to_img(t):
    a = (t.numpy().transpose(1, 2, 0) * 255).clip(0, 255).astype("uint8")
    return Image.fromarray(a)

print("俯视 up：")
display(to_img(sample["observation.images.up"]))
print("侧视 side：")
display(to_img(sample["observation.images.side"]))

### 播放一条轨迹（动图）
把某条轨迹的两路相机拼在一起做成动图。你会看到机械臂 **张爪 → 下探 → 夹起乐高 → 搬到盒子上方 → 松爪** 的完整过程。
（用 GIF 是为了在任何环境都能内联播放，不依赖视频编码器。生成需十几秒。）

In [ ]:
ep = 0                                   # 想看别的轨迹就改这里（0 ~ 49）
row = meta.episodes[ep]
start, end = int(row["dataset_from_index"]), int(row["dataset_to_index"])

frames = []
for gi in range(start, end, 4):          # 每 4 帧取 1 帧，加快生成
    s = explore_ds[gi]
    up = to_img(s["observation.images.up"]).resize((320, 240))
    side = to_img(s["observation.images.side"]).resize((320, 240))
    combo = Image.new("RGB", (640, 240))
    combo.paste(up, (0, 0)); combo.paste(side, (320, 0))
    frames.append(combo)

frames[0].save("demo_episode.gif", save_all=True, append_images=frames[1:], duration=80, loop=0)
print(f"轨迹 {ep}：{end - start} 帧 | 左 = 俯视，右 = 侧视")
IPyImage("demo_episode.gif")

### 看动作数据（6 个关节随时间）
机器人每一时刻的动作 $=$ **6 个关节角度**。把这条轨迹的动作画成曲线——
重点看 **`gripper`（夹爪）** 那条线：**升高 $=$ 张开、降低 $=$ 闭合**，
正好对应「张爪 → 下探 → 夹住 → 搬运 → 松开」的抓取过程。

**观察题**：`gripper` 第一次明显张开大约在第几秒？那一刻机械臂在做什么？

In [ ]:
acts = np.stack([explore_ds[gi]["action"].numpy() for gi in range(start, end)])
t = np.arange(len(acts)) / meta.fps

plt.figure(figsize=(8, 4))
for j, name in enumerate(meta.names["action"]):
    plt.plot(t, acts[:, j], label=name, linewidth=1.3)
plt.xlabel("时间（秒）"); plt.ylabel("关节位置")
plt.title(f"轨迹 {ep} 的动作（6 个关节）"); plt.legend(fontsize=8, ncol=3)
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## 第二部分：ACT 的网络结构

ACT $=$ **CVAE（条件变分自编码器）$+$ Transformer 编解码器**，总参数约 **51.6 M**。一句话：
**看 1 帧双目画面 $+$ 关节状态，一次性输出未来 100 步动作（动作分块）。**

### 输入（单个样本，记 `B` $=$ batch）
| 名称 | 形状 | 含义 |
|---|---|---|
| `observation.images.up` | `(3,480,640)` | 俯视相机 RGB |
| `observation.images.side` | `(3,480,640)` | 侧视相机 RGB |
| `observation.state` | `(6,)` | 当前 6 个关节角 |
| `action`（仅训练） | `(100,6)` | 目标：未来 100 步 × 6 关节 |

### ① 视觉编码：ResNet18 骨干（11.2 M，ImageNet 预训练）
- 每路相机 `(3,480,640)` $\to$ ResNet18 `layer4` $\to$ 特征图 `(512,15,20)`
  （**下采样 32 倍**：$480/32{=}15,\ 640/32{=}20$）
- `1×1` 卷积投影（0.26 M）：`(512,15,20)→(512,15,20)`，把视觉通道**对齐到** $d_{\text{model}}{=}512$
- 摊平 $15{\times}20{=}300$ $\to$ **每路 300 个 token**，两路共 **600 个视觉 token**

### ② CVAE 编码器：BERT 式（17.3 M，**仅训练时用**）
- 输入序列 `[CLS, state, 100个动作]` $=102$ 个 token $\to$ 4 层**双向** Transformer Encoder
- 取 `CLS` 的输出 $\to$ 线性层 $\to$ $\mu,\log\sigma^2$（`latent_dim=32`）$\to$ 重参数化采样得 $z\in\mathbb R^{32}$
- **推理时跳过整块，直接 $z=0$**

### ③ Transformer 编码器（17.3 M，4 层）$+$ 解码器（5.4 M，1 层）
- 拼装 **602 个 token**：$1$（$z$）$+1$（state）$+600$（视觉）$\to$ `(602,B,512)`
- 编码器：`(602,B,512) → (602,B,512)`
- 解码器：**100 个可学习位置编码**当 query（输入是全 0），cross-attention 到编码器输出 $\to$ `(100,B,512)`

### ④ 动作头（Linear，约 3 K）
- `(B,100,512) → (B,100,6)`，即 **100 步 × 6 关节** 的动作块

### 损失
$$\mathcal L = \underbrace{\ell_1(\hat a, a)}_{\text{动作重构}} \; + \; \beta\cdot \underbrace{\mathrm{KL}(\mu,\log\sigma^2)}_{\text{CVAE 正则}},\qquad \beta = 10.$$

下面建好策略后，我们会**打印真实的参数量**来印证上表。

---
## 第三部分：准备数据与策略

### 任务 1：划分输入 / 输出特征
数据集的 metadata 告诉策略「输入是什么、输出是什么」（就像 CartPole 里 `env` 定义观测/动作空间）。
其中**动作**特征作为输出，其余（状态 + 图像）作为输入。

**提示**：动作特征的 `ft.type` 等于 `FeatureType.ACTION`。

In [ ]:
features = dataset_to_policy_features(meta.features)

# TODO（填空 1）：把判断条件补全，选出“动作”特征作为输出
output_features = {k: ft for k, ft in features.items() if ...}        # ← 修改 ...
input_features = {k: ft for k, ft in features.items() if k not in output_features}

print("输入特征:", list(input_features))
print("输出特征:", list(output_features))

### 任务 2：构建 ACT 配置与策略
把输入/输出特征传给 `ACTConfig`，再用它实例化 `ACTPolicy`。
`make_pre_post_processors` 负责用数据集统计量做**归一化**（MEAN/STD）。

**提示**：`ACTConfig(input_features=?, output_features=?)`。

In [ ]:
# TODO（填空 2）：用上面的特征构建 ACT 配置
cfg = ACTConfig(input_features=..., output_features=...)              # ← 填入两个参数

policy = ACTPolicy(cfg).to(device)
preprocess, postprocess = make_pre_post_processors(cfg, dataset_stats=meta.stats)

print("动作分块长度 chunk_size =", cfg.chunk_size)
print("Transformer 宽度 dim_model =", cfg.dim_model, "| 潜变量维度 latent_dim =", cfg.latent_dim)

### 任务 3：动作分块的时间戳
ACT 要预测**未来 100 步**动作，所以数据集每个样本的 `action` 必须返回一段 100 步的序列。
`make_delta` 把「索引」转成「时间戳」（秒）。动作要用 `cfg.action_delta_indices`（就是 `[0,1,…,99]`）。

**提示**：第一个参数填 `cfg.action_delta_indices`。

In [ ]:
def make_delta(idx, fps):
    return [0] if idx is None else [i / fps for i in idx]

# TODO（填空 3）：让 action 返回未来 chunk_size 步（填第一个参数）
delta = {"action": make_delta(..., meta.fps)}                         # ← 填入 ...
delta |= {k: make_delta(cfg.observation_delta_indices, meta.fps) for k in cfg.image_features}

dataset = LeRobotDataset(dataset_id, delta_timestamps=delta, video_backend="pyav")
loader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0, drop_last=True)

batch0 = preprocess(next(iter(loader)))
print("action 目标形状 (B, T, dim):", tuple(batch0[ACTION].shape))   # 应为 (8, 100, 6)

### 观察网络结构（无需填空）
建好策略后，逐模块统计参数量，印证第二部分的结构表。

In [ ]:
def count(m):
    return sum(p.numel() for p in m.parameters())

total = count(policy.model)
print(f"{'组件':<34}{'参数量':>14}{'占比':>9}")
print("-" * 58)
for name, child in policy.model.named_children():
    n = count(child)
    if n:
        print(f"{name:<34}{n:>14,}{100*n/total:>8.1f}%")
print("-" * 58)
print(f"{'合计':<34}{total:>14,}")

---
## 第四部分：训练 ACT（行为克隆）

### 任务 4：补全训练循环
ACT 的损失是**监督式**的：`policy.forward(batch)` 直接返回 $\ell_1+\mathrm{KL}$。
训练循环和 CartPole 那个一样，区别是「奖励/回报」换成了「监督损失」。

你要补全标准的**反向传播三步**：清梯度 → 反向 → 更新。

**提示**：`optimizer.zero_grad()` / `loss.backward()` / `optimizer.step()`。

In [ ]:
def batches(loader):
    """无限地产出 batch（循环遍历 loader）。"""
    while True:
        for b in loader:
            yield b
stream = batches(loader)

def to_device(b):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}

policy.train()
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-4)
training_steps = 300
loss_history = []

for step in range(1, training_steps + 1):
    batch = to_device(preprocess(next(stream)))
    loss, _ = policy.forward(batch)          # 已给：ACT 的 L1 + KL 行为克隆损失

    # TODO（填空 4）：写出反向传播三步
    ...                                      # ← 清空梯度
    ...                                      # ← 反向传播
    ...                                      # ← 更新参数

    loss_history.append(loss.item())
    if step % 25 == 0:
        print(f"训练步 {step:4d} | 损失 {loss.item():7.3f}")
print("行为克隆训练完成。")

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(loss_history)
plt.xlabel("训练步"); plt.ylabel("ACT 损失 (L1 + KL)")
plt.title("行为克隆：损失下降曲线"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## 第五部分：使用训练好的策略（推理）

### 任务 5：预测一个动作块
推理时把策略切到 `eval()`，CVAE 潜变量取 $z=0$，输出是**确定性**的动作块。

**提示**：用 `policy.predict_action_chunk(...)`。

In [ ]:
policy.reset(); policy.eval()
obs = to_device(preprocess(next(stream)))
obs_infer = {k: (v[:1] if torch.is_tensor(v) and v.dim() > 0 else v)
             for k, v in obs.items() if k != ACTION and not k.endswith("_is_pad")}

with torch.no_grad():
    # TODO（填空 5）：用训练好的策略预测一个动作块
    chunk = ...                               # ← 调用 policy 的推理方法

print("预测的动作块形状:", tuple(chunk.shape), " = (1, chunk_size, action_dim)")
print("第 0 步预测动作（6 关节）:", chunk[0, 0].cpu().numpy().round(2))

---
## 小结

- **行为克隆**：从专家示范中监督式地学「观测 → 动作块」，损失是 $\ell_1+\mathrm{KL}$，**不需要奖励**。
- **ACT 网络**：ResNet18 视觉骨干（把图像变成 600 个 token）$+$ Transformer 编解码器（一次输出 100 步动作）$+$ CVAE（训练时把动作压成风格变量 $z$，推理时 $z{=}0$）。
- **动作分块**：一次预测 100 步 $\Rightarrow$ 决策点更少、更平滑，是 ACT 区别于普通单步 BC 的关键。

### 思考题（选做）
1. 把 `training_steps` 调到几千，loss 还能降到多少？动作块预测会更接近专家吗？
2. 如果把 `chunk_size` 改成 50，网络结构里哪些形状会变？参数量变化大吗？
3. 推理时若**不**取 $z=0$ 而是从 $\mathcal N(0,I)$ 采样 $z$，输出会怎样？这对「探索」有什么用？